In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt

# ==========================================
# 1. Load Processed Data
# ==========================================
# We use the file we exported from the previous step
df = pd.read_csv('../data/train_FD001_processed.csv')

# ==========================================
# 2. Feature & Target Selection
# ==========================================
# X: All columns except the ID (unit_nr), time index, and the target (RUL)
X = df.drop(columns=['unit_nr', 'time_cycles', 'RUL'])
# y: The target we want to predict (Remaining Useful Life)
y = df['RUL']

# ==========================================
# 3. Train/Validation Split
# ==========================================
# We split the data: 80% for training, 20% for testing our model's accuracy
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Validation features shape: {X_val.shape}")

Training features shape: (16504, 53)
Validation features shape: (4127, 53)


In [2]:
# ==========================================
# 4. Baseline Model Training (Linear Regression)
# ==========================================
model = LinearRegression()
model.fit(X_train, y_train)

# ==========================================
# 5. Prediction & Evaluation
# ==========================================
y_pred = model.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f"Baseline Linear Regression Results:")
print(f"-----------------------------------")
print(f"RMSE: {rmse:.2f} cycles")
print(f"MAE:  {mae:.2f} cycles")
print(f"R2 Score: {r2:.4f}")

Baseline Linear Regression Results:
-----------------------------------
RMSE: 42.77 cycles
MAE:  32.78 cycles
R2 Score: 0.5997


In [3]:
from sklearn.ensemble import RandomForestRegressor

# ==========================================
# 6. Advanced Model Training (Random Forest)
# ==========================================
# n_estimators=100: We use 100 decision trees to vote on the RUL
# max_depth=15: To prevent overfitting (keep the model generalizable)
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)

print("Training Random Forest Regressor... Please wait.")
rf_model.fit(X_train, y_train)

# ==========================================
# 7. Evaluation
# ==========================================
y_pred_rf = rf_model.predict(X_val)

rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_rf))
mae_rf = mean_absolute_error(y_val, y_pred_rf)
r2_rf = r2_score(y_val, y_pred_rf)

print(f"\nRandom Forest Results:")
print(f"----------------------")
print(f"RMSE: {rmse_rf:.2f} cycles")
print(f"MAE:  {mae_rf:.2f} cycles")
print(f"R2 Score: {r2_rf:.4f}")

Training Random Forest Regressor... Please wait.

Random Forest Results:
----------------------
RMSE: 20.81 cycles
MAE:  13.92 cycles
R2 Score: 0.9052


In [5]:
import joblib
import os

# ==========================================
# 8. Saving the Model and Scaler for Production
# ==========================================
# Create the models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save the trained Random Forest model
model_path = '../models/random_forest_v1.joblib'
joblib.dump(rf_model, model_path)

# IMPORTANT: We must save the scaler used during training
# Without the exact same scaler, the production API will fail


print(f"[SUCCESS] Model saved to: {model_path}")
print(f"[SUCCESS] Scaler saved to: {scaler_path}")

[SUCCESS] Model saved to: ../models/random_forest_v1.joblib
[SUCCESS] Scaler saved to: ../models/scaler.joblib
